In [1]:
# %pip uninstall pymysql

^C
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# %pip install html5lib

In [17]:
from datetime import date, datetime, timedelta
from threading import Timer
import pandas as pd
import requests
from bs4 import BeautifulSoup
import pymysql, calendar, time, json
from io import StringIO

In [18]:
import warnings

warnings.filterwarnings("ignore")

In [ ]:
class CollecData:
    def __init__(self): # 생성자
        # Naver API 설정
        self.url = 'https://m.stock.naver.com/api/stock/{}/price'
        self.headers = {'User-Agent': 'Mozilla/5.0'}

        # 데이터 수집 설정
        self.years = 3
        self.page_size = 60
        self.sleep = 0.3

        # 종목코드 딕셔너리
        self.codes = {}

        # Timer 초기화
        self.timer = None

        self.create_tables()

    ########### 데이터 테이블 생성 ########### 
    # krx data, naver 금융 data 저장 테이블 생성
    def create_tables(self):
        self.conn = pymysql.connect(host='localhost', user='root', \
            password='1111', database='INVESTAR', charset='utf8')

        with self.conn.cursor() as curs:
            sql = """
                CREATE TABLE IF NOT EXISTS company_info (
                    code VARCHAR(20),
                    company VARCHAR(40),
                    last_update DATE,
                    PRIMARY KEY (code))
                """
            curs.execute(sql)

            sql = """
            CREATE TABLE IF NOT EXISTS daily_price (
                code VARCHAR(20),
                date DATE,
                open BIGINT(20),
                high BIGINT(20),
                low BIGINT(20),
                close BIGINT(20),
                diff BIGINT(20),
                volume BIGINT(20),
                PRIMARY KEY (code, date))
            """
            curs.execute(sql)

        self.conn.commit()  
        self.codes = dict()   

        
    #################################################
    ###########  krx에서 데이터 수집 및 저장 ########### 
    #################################################
    def read_krx_code(self): # DataFrame(return)
        url = "http://kind.krx.co.kr/corpgeneral/corpList.do?method=download&searchType=1"
        krx = pd.read_html(url, header=0, encoding="cp949")[0]
        krx = krx[['종목코드', '회사명']]
        krx = krx.rename(columns={'종목코드': 'code', '회사명': 'company'})
        krx.code = krx.code.str.zfill(6)

        return krx

    # krx 크롤링된 데이터 저장(MySQL)
    def update_comp_info(self):
        sql = "SELECT * FROM company_info"
        df = pd.read_sql(sql, self.conn)
        for idx in range(len(df)):
            self.codes[df['code'].values[idx]] = df['company'].values[idx]

        with self.conn.cursor() as curs:    
            sql = "SELECT max(last_update) FROM company_info"
            curs.execute(sql)    
            rs = curs.fetchone()

            # 당일 날짜에 수집후 저장된 내용이 없다면 수집 저장하고 아니면 pass
            today = datetime.today().strftime('%Y-%m-%d') # 09/02/26
            if rs[0] == None or rs[0].strftime('%Y-%m-%d') < today:
                krx = self.read_krx_code()

                for idx in range(len(krx)):
                    code = krx.code.values[idx]
                    company = krx.company.values[idx]   

                    # REPLACE INTO : 키가 중복되면 기존 행을 삭제하고 새로운 행으로 교체
                    sql = f"REPLACE INTO company_info (code, company, last"\
                            f"_update) VALUES ('{code}', '{company}', '{today}')"
                    curs.execute(sql)

                    tmnow = datetime.now().strftime('%Y-%m-%d %H:%M')
                    print(f"[{tmnow}] #{idx+1:04d} REPLACE INTO company_info "\
                                f"VALUES ({code}, {company}, {today})")  
                    
                self.conn.commit()                 
            
    ###############################################
    ########### 네이버 데이터 수집 및 저장 ########### 
    ###############################################
    # 문자열을 숫자로 변환
    def to_int(self, value):
        if value is None or value == "":
            return 0

        return int(str(value).replace(",", "").replace("+", ""))

    # 네이버 주식 거래 정보 수집
    def read_naver(self, code, company, pages_to_fetch):
        rows = []

        # 최근 3년 데이터
        start_date = date.today() - timedelta(
            days=365 * self.years
        )

        for page in range(1, pages_to_fetch + 1):

            try:

                res = requests.get(self.url.format(code),
                    params={
                        'pageSize': self.page_size,
                        'page': page
                    },
                    headers=self.headers,
                    timeout=10
                )

                res.raise_for_status()

                items = res.json()
                if not items:
                    break

                stop = False
                for item in items:
                    traded_date = datetime.strptime(
                        item['localTradedAt'][:10],
                        '%Y-%m-%d'
                    ).date()

                    # 3년 이전 데이터 수집 중단
                    if traded_date < start_date:

                        stop = True
                        break

                    # 전일 대비 가격 차이
                    diff = self.to_int(
                        item.get(
                            'compareToPreviousClosePrice'
                        )
                    )

                    direction = (
                        item.get(
                            'compareToPreviousPrice'
                        ) or {}
                    ).get('name', '')

                    # 하락 및 하한가 음수 처리
                    if direction in (
                        'FALLING',
                        'LOWER_LIMIT'
                    ) and diff > 0:

                        diff = -diff

                    rows.append({
                        'code': code,
                        'date': traded_date,
                        'open': self.to_int(
                            item.get('openPrice')
                        ),
                        'high': self.to_int(
                            item.get('highPrice')
                        ),
                        'low': self.to_int(
                            item.get('lowPrice')
                        ),
                        'close': self.to_int(
                            item.get('closePrice')
                        ),
                        'diff': diff,
                        'volume': self.to_int(
                            item.get(
                                'accumulatedTradingVolume'
                            )
                        )
                    })

                # 과거 데이터 또는 마지막 페이지 확인
                if stop or len(items) < self.page_size:
                    break

                time.sleep(self.sleep)

            except requests.RequestException as e:
                print(f"{company} ({code}) "  f"네이버 수집 오류: {e}" )
                break

            except (ValueError, KeyError, TypeError) as e:
                print(f"{company} ({code}) "  f"데이터 변환 오류: {e}")
                break

        df = pd.DataFrame(rows)

        print(
            f"{company} ({code}) "
            f"데이터 수집: {len(df)}건"
        )

        return df

    # Naver 주식 거래 정보 DB 저장(한 개의 회사)
    def replace_into_db(self, df, num, code, company):  
        if df.empty:
            print( f"{company} ({code}): 저장할 데이터가 없습니다." )
            return

        # DUPLICATE KEY UPDATE : data가 없으면 insert, 있으면 update
        sql = """
            INSERT INTO daily_price(code, date, open, high, low, close, diff, volume)  \
                VALUES (%s, %s, %s, %s, %s, %s, %s, %s) ON DUPLICATE KEY UPDATE 
                open = VALUES(open), high = VALUES(high), low = VALUES(low), close = VALUES(close), \
                diff = VALUES(diff), volume = VALUES(volume)
            """
        
        data = [ ( row.code, row.date, row.open, row.high, row.low,  row.close,  row.diff, row.volume )  
                for row in df.itertuples(index=False)]
        with self.conn.cursor() as cur:
            cur.executemany(sql, data)

        self.conn.commit()
        print(f"{company} ({code}) : "   f"{len(data)}건 DB 저장 완료")

    # 일별 주가 데이터 업데이트 (전체 회사)
    def update_daily_price(self, pages_to_fetch):
        if not self.codes:
            print("수집할 종목이 없습니다.")
            return
        
        # start=1 : inx를 1부터 시작 
        for idx, (code, company) in enumerate(self.codes.items(), start=1):
            # if code != '030190' & code != '005930' & code != '024100' & code != '017670' & code != '012450':  # 삼성전자, NICE평가정보
            if code not in ['030190', '005930', '086790', '017670', '012450']:
                continue

            print(f"\n[{idx}/{len(self.codes)}] " f"[{company}] 데이터 수집 시작")
            df = self.read_naver(code, company, pages_to_fetch)
            self.replace_into_db(df, len(df), code, company)
    
    # DB 연결 종료
    def close(self):
        # hasattr(self, 'conn' : 객체에(self) 특정 속성이(conn) 존재하는지 확인
        if hasattr(self, 'conn') and self.conn.open:
            self.conn.close()
            print("DB 연결 종료")        

    ########### Entry function ###########
    def execute_daily(self):
        self.update_comp_info()

        # config.json 파일 읽기
        try:
            with open('config.json', 'r') as in_file:
                config = json.load(in_file)
                pages_to_fetch = config.get('pages_to_fetch', 100)
        except FileNotFoundError:
            pages_to_fetch = 100
            config = {
                'pages_to_fetch': pages_to_fetch
            }

            with open('config.json', 'w') as out_file:
                json.dump(config, out_file, indent=4)  # indent=4 : 들여쓰기 4칸

        self.update_daily_price(pages_to_fetch)

        # Timer 설정
        tmnow = datetime.now()
        lastday = calendar.monthrange(tmnow.year, tmnow.month)[1]  # 해당 월을 말일 

        if tmnow.month == 12 and tmnow.day == lastday:
            tmnext = tmnow.replace(year=tmnow.year + 1, month=1, day=1,  hour=17, minute=0, second=0, microsecond=0)
        elif tmnow.day == lastday:
                tmnext = tmnow.replace(month=tmnow.month + 1, day=1, hour=17, minute=0, second=0, microsecond=0)
        else:
            tmnext = tmnow.replace(day=tmnow.day + 1, hour=17, minute=0, second=0, microsecond=0)

        # 현재 시각부터 다음 실행 시각까지 남은 초를 구하되, 최소 1초 이상이 되도록 보정
        tmdiff = tmnext - tmnow
        secs = max(1, int(tmdiff.total_seconds()))

         # Timer 생성
        self.timer = Timer(secs, self.execute_daily)
        print("Waiting for next update ({}) ... ".format(
                    tmnext.strftime('%Y-%m-%d %H:%M')
            )
        )

        # 다음 날 자동 실행
        self.timer.daemon = True
        self.timer.start()

# 소멸자
def __del__(self):
    # 이전 타이머가 남아있으면 취소 (중복 방지)
    if getattr(self, 'timer', None) is not None:
        self.timer.cancel()

    if hasattr(self, 'conn') and self.conn.open:
        self.conn.close()


In [ ]:
'''  
= 프로그램 자동실행을 위한 windows registry 등록
1. 윈도우 키에서 regedit(레지스트리 편집기) 실행
2. HKEY_CURRENT_USER\Software\Microsoft\Windows\CurrentVersion\Run 이동
3. 이름 : CollecData, 데이터 :  C:\work\stock\CollecData.py
'''

In [36]:
dbu = CollecData()
# krx = dbu.update_comp_info()
# dbu.read_naver('005930', '삼성전자', 100)
dbu.execute_daily()


[282/2768] [삼성전자] 데이터 수집 시작
삼성전자 (005930) 데이터 수집: 730건
삼성전자 (005930) : 730건 DB 저장 완료

[467/2768] [한화에어로스페이스] 데이터 수집 시작
한화에어로스페이스 (012450) 데이터 수집: 730건
한화에어로스페이스 (012450) : 730건 DB 저장 완료

[566/2768] [SK텔레콤] 데이터 수집 시작
SK텔레콤 (017670) 데이터 수집: 730건
SK텔레콤 (017670) : 730건 DB 저장 완료

[704/2768] [NICE평가정보] 데이터 수집 시작
NICE평가정보 (030190) 데이터 수집: 730건
NICE평가정보 (030190) : 730건 DB 저장 완료

[1331/2768] [하나금융지주] 데이터 수집 시작
하나금융지주 (086790) 데이터 수집: 730건
하나금융지주 (086790) : 730건 DB 저장 완료


In [37]:
krx

In [22]:
# test code
from datetime import datetime

starttime = datetime.now()
print(starttime)

print(starttime.strftime("%d"))
print(starttime.strftime("%D"))
print(starttime.strftime("%m/%d/%y"))
print(starttime.strftime("%m"))
print(starttime.strftime("%M"))
print(starttime.strftime("%Y"))
print(starttime.strftime("%y"))

print(starttime.strftime("%Y-%m-%d %H:%M:%S.%f")[:-3])  # %f : 마이크로초 6자리

2026-11-13 13:17:37.736073
13
11/13/26
11/13/26
11
17
2026
26
2026-11-13 13:17:37.736
